In [ ]:
import os
import yaml
import pytz
import requests
from pathlib import Path
import numpy as np
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

# import functions from bookie_grabber
from bookie_grabber import *

from bookie_emailer import email_workbook

load_dotenv()

In [ ]:
os.path.exists(Path("/Users/notbahd/Desktop/BookieGrabber/data/ready/sent"))

In [ ]:
ready_games_workbook_path = Path("/Users/notbahd/Desktop/BookieGrabber/data/ready/ready_games_2025-12-22_193601.xlsx")

In [ ]:

if os.path.exists(ready_games_workbook_path):
    success = email_workbook(
        ready_games_workbook_path,
        subject=f"Ready Games – test",
        body="Attached is today's ready games workbook.",
    )
    if not success:
        print("failed to send")

In [ ]:
# ┌─────────────────────────────────────────────────────────┐
# │  1. SPORTS                                              │
# │  What sports are available?                             │
# │  GET /v3/sports                                         │
# │  Returns: Football, Basketball, Tennis, etc.            │
# └──────────────────┬──────────────────────────────────────┘
#                    │
#                    ↓
# ┌─────────────────────────────────────────────────────────┐
# │  2. LEAGUES                                             │
# │  What leagues exist in a sport?                         │
# │  GET /v3/leagues?sport=football                         │
# │  Returns: Premier League, La Liga, Bundesliga, etc.     │
# └──────────────────┬──────────────────────────────────────┘
#                    │
#                    ↓
# ┌─────────────────────────────────────────────────────────┐
# │  3. EVENTS                                              │
# │  What matches are coming up?                            │
# │  GET /v3/events?sport=football&league=premier-league    │
# │  Returns: Man Utd vs Liverpool, Chelsea vs Arsenal, etc.│
# └──────────────────┬──────────────────────────────────────┘
#                    │
#                    ↓
# ┌─────────────────────────────────────────────────────────┐
# │  4. ODDS                                                │
# │  What are the betting odds?                             │
# │  GET /v3/odds?eventId=123456&bookmakers=Bet365,Unibet   │
# │  Returns: Odds from multiple bookmakers                 │
# └─────────────────────────────────────────────────────────┘

In [ ]:
# "status": "pending" for events
# Example usage
df_totals, df_btts = main()

# Map New League

to map

    turkiye-super-lig - Turkish Super Cup

In [ ]:
api_key = load_env()
events = get_league_events(api_key, "scotland-premiership", limit=100)
dfevents = pd.DataFrame(events)
sorted(dfevents.away.unique().tolist() + dfevents.home.unique().tolist())
# sorted(dfevents.away.unique().tolist())

In [ ]:
"""
AUTHOR: JZ
DATE: 4 Dec 2025 
DESCRIPTION: Betfair API integration to fetch Over/Under market volumes for specified leagues.
"""

import os
import json
import yaml
import pandas as pd
import requests
from dotenv import load_dotenv

# ---------------------------
# Load .env credentials
# ---------------------------
load_dotenv()

APP_KEY = os.getenv("BETFAIR_API_KEY")
USERNAME = os.getenv("BETFAIR_USERNAME")
PASSWORD = os.getenv("BETFAIR_PASSWORD")

CERT_FILE = "client-2048.crt"
KEY_FILE = "client-2048.key"

BETFAIR_API_URL = "https://api.betfair.com/exchange/betting/json-rpc/v1"
SSO_CERT_URL = "https://identitysso-cert.betfair.com/api/certlogin"

MARKETS = ["OVER_UNDER_15", "OVER_UNDER_25", "OVER_UNDER_35", "BOTH_TEAMS_TO_SCORE"]

# ---------------------------
# Load leagues from YAML
# ---------------------------
with open("config.yaml", "r") as f:
    CONFIG = yaml.safe_load(f)

LEAGUES = CONFIG.get("leagues", [])

# ---------------------------
# JSON-RPC helper
# ---------------------------
def make_request(app_key:str, session_token:str, payload):
    headers = {
        "X-Application": app_key,
        "X-Authentication": session_token,
        "Content-Type": "application/json"
    }
    response = requests.post(BETFAIR_API_URL, data=json.dumps(payload), headers=headers)
    response.raise_for_status()
    return response.json()

# ---------------------------
# Betfair certificate login
# ---------------------------
def get_session_token():
    data = {"username": USERNAME, "password": PASSWORD}
    headers = {"X-Application": APP_KEY, "Content-Type": "application/x-www-form-urlencoded"}

    response = requests.post(SSO_CERT_URL, data=data, headers=headers, cert=(CERT_FILE, KEY_FILE))
    result = response.json()

    if result.get("loginStatus") == "SUCCESS":
        return result["sessionToken"]
    else:
        raise Exception("Login failed: " + result.get("loginStatus", "Unknown error"))

# ---------------------------
# Generic league functions
# ---------------------------
def get_competition_id(session_token:str, league_name):
    payload = {
        "jsonrpc": "2.0",
        "method": "SportsAPING/v1.0/listCompetitions",
        "params": {"filter": {"eventTypeIds": ["1"]}},  # 1 = Soccer
        "id": 1
    }

    data = make_request(APP_KEY, session_token, payload)

    for item in data["result"]:
        if item["competition"]["name"].lower() == league_name.lower():
            return item["competition"]["id"]

    raise ValueError(f"Competition '{league_name}' not found.")

def get_over_under_markets(session_token:str, competition_id):
    payload = {
        "jsonrpc": "2.0",
        "method": "SportsAPING/v1.0/listMarketCatalogue",
        "params": {
            "filter": {
                "competitionIds": [competition_id],
                "marketTypeCodes": ["OVER_UNDER_15", "OVER_UNDER_25", "OVER_UNDER_35", "BOTH_TEAMS_TO_SCORE", "MATCH_ODDS"],
            },
            "maxResults": "200",
            "marketProjection": ["EVENT", "MARKET_DESCRIPTION", "RUNNER_DESCRIPTION"]
        },
        "id": 1
    }

    data = make_request(APP_KEY, session_token, payload)
    return data["result"]

def get_ou_volume(session_token:str, league_name:str):
    competition_id = get_competition_id(session_token, league_name)
    markets = get_over_under_markets(session_token, competition_id)

    rows = []
    for m in markets:
        rows.append({
            "marketId": m["marketId"],
            "line": m["marketName"],
            "total_volume": m["totalMatched"],
            "event": m["event"]["name"],
        })

    return pd.DataFrame(rows)

# ---------------------------
# Example usage
# ---------------------------
# if __name__ == "__main__":
#     session_token = get_session_token()
#     print("SESSION TOKEN:", session_token)

#     # Pick a league from config (for example, first league)
#     league_name = LEAGUES[0]["name"]
#     df = get_ou_volume(session_token, league_name)
#     print(df.head())


In [ ]:
session_token = get_session_token()
print("SESSION TOKEN:", session_token)
# Pick a league from config (for example, first league)
league_name = "Scottish Premiership" 
df = get_ou_volume(session_token, league_name)
df.event.unique()

In [ ]:
from openpyxl import load_workbook
import yaml



def load_yaml_safe(path: Path):
    if not path.exists():
        return {}
    try:
        with path.open("r", encoding="utf-8") as f:
            return yaml.safe_load(f) or {}
    except Exception as e:
        print(f"Failed to load YAML {path}: {e}")
        return {}

def expand_formula(template: str, row: int, bf_col: str | None = None, rpd_col: str | None = None) -> str:
    """
    Expands an Excel formula template for a given row:
        - replaces cell refs (L2/R2/H2/T2/I2/J2) with the correct row
        - replaces {ODDS} 
        - applies any custom substitutions
    """
    # Replace generic cell refs
    formula = (
        template
        .replace("L2", f"L{row}")
        .replace("R2", f"R{row}")
        .replace("H2", f"H{row}")
        .replace("T2", f"T{row}")
        .replace("I2", f"I{row}")
        .replace("J2", f"J{row}")
    )
    # Replace {ODDS}
    if bf_col and rpd_col:
        formula = formula.replace("{ODDS}", bf_col).replace("{RPD}", rpd_col)


    print("Expanded formula:", formula)
    return formula

def apply_excel_formulas(workbook_path: Path, config: dict):
    wb = load_workbook(workbook_path)
    # -------- TOTALS --------
    if "totals_ready" in wb.sheetnames:
        ws = wb["totals_ready"]
        ws["T1"] = "o/u"
        ws["U1"] = "stake"
        for row in range(2, ws.max_row + 1):
            # O/U formula
            ws[f"T{row}"] = expand_formula(config["excel_formulas"]["O/U"], row)
            over_rpd = float(ws[f"K{row}"].value)
            under_rpd = float(ws[f"L{row}"].value)
            if over_rpd < under_rpd:
                bf_col = f"I{row}"
                rpd_col = f"K{row}"
            else:
                bf_col = f"J{row}"
                rpd_col = f"L{row}"
            print(over_rpd, under_rpd, over_rpd < under_rpd, bf_col, rpd_col)

            line = ws[f"Q{row}"].value
            if line == "Over/Under 1.5 Goals":
                ws[f"U{row}"] = expand_formula(config["excel_formulas"]["totals"]["g1_5"], row, bf_col, rpd_col)
            elif line == "Over/Under 2.5 Goals":
                ws[f"U{row}"] = expand_formula(config["excel_formulas"]["totals"]["g2_5"], row, bf_col, rpd_col)
            else:
                ws[f"U{row}"] = expand_formula(config["excel_formulas"]["totals"]["g3_5"], row, bf_col, rpd_col)
    # -------- BTTS --------
    if "btts_ready" in wb.sheetnames:
        ws = wb["btts_ready"]
        ws["T1"] = "o/u"
        ws["U1"] = "stake"
        for row in range(2, ws.max_row + 1):
            # O/U formula
            ws[f"T{row}"] = expand_formula(config["excel_formulas"]["O/U"], row)
            over_rpd = ws[f"K{row}"].value
            under_rpd = ws[f"L{row}"].value
            # bf_col = f"J{row}" if over_rpd < under_rpd else f"I{row}"
            if over_rpd < under_rpd:
                bf_col = f"J{row}"
                rpd_col = f"K{row}"
            else:
                bf_col = f"I{row}"
                rpd_col = f"L{row}"
            print(over_rpd, under_rpd, over_rpd < under_rpd, bf_col, rpd_col)
            # BTTS stake formula
            ws[f"U{row}"] = expand_formula(config["excel_formulas"]["btts"]["stake"], row, bf_col, rpd_col)

    wb.save(workbook_path)

formulas = load_yaml_safe(Path(r"formulas_test.yaml"))
ready_games_workbook_path = r"/Users/notbahd/Desktop/ready_games_german_bundesliga_2026-01-13_162011.xlsx"
apply_excel_formulas(ready_games_workbook_path, formulas)

In [ ]:
from datetime import datetime, timedelta, timezone

end_dt = (
    datetime.now(timezone.utc)
    .date() + timedelta(days=2)
)

end_rfc3339 = datetime(
    end_dt.year,
    end_dt.month,
    end_dt.day,
    23, 59, 59,
    tzinfo=timezone.utc
).isoformat().replace("+00:00", "Z")

print(end_rfc3339)


In [ ]:
API_BASE = "https://api2.odds-api.io/v3"

def load_env():
    load_dotenv()
    api_key = os.getenv("ODDS_API_KEY")
    if not api_key:
        raise ValueError("ODDS_API_KEY not found in .env file")
    return api_key


def get_league_events(api_key: str, league: str, limit: int = 200):
    """Fetch list of upcoming events for a given league from Odds-API."""
    url = f"{API_BASE}/events"
    # "status": "pending"
    end_dt = (
            datetime.now(timezone.utc)
            .date() + timedelta(days=2)
        )

    end_rfc3339 = datetime(
            end_dt.year,
            end_dt.month,
            end_dt.day,
            23, 59, 59,
            tzinfo=timezone.utc
        ).isoformat().replace("+00:00", "Z")

    params = {"apiKey": api_key, "sport": "football", "league": league, "limit": limit, "to": end_rfc3339}
    try:
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        data = r.json()
        
        # The API might put the list under data or return a list directly
        # Try common fields first:
        if isinstance(data, dict):
            for key in ("data", "events", "results"):
                if key in data:
                    return data[key]
            # fallback: maybe top-level "data" missing but content is in 'response'
            return data.get("data", [])
        return data
    except Exception as e:
        print(f"[ERROR] Failed to fetch league events for {league}: {e}")
        return []
    
api_key = load_env()
events = get_league_events(api_key, "england-premier-league", limit=10)

In [ ]:
len(events)